# Preprocessing on High Volume For-Hire Vehicle (HVFHV) Trip Records Dataset:

In this notebook, we are mainly focusing on convertint textual data to numerical data, and drop unused columns.

----

# Import Libraries:

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
from pyspark.sql.functions import when, col
import os

In [2]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("preprocessing_hvfhv")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

24/08/21 03:05:28 WARN Utils: Your hostname, Cocos-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.16.33.67 instead (on interface en0)
24/08/21 03:05:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/21 03:05:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/08/21 03:05:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/08/21 03:05:29 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


# Read Files:

Read Preprocessed HVFHV Parquet Files:

In [3]:
base_dir = "../data"

In [4]:
hvfhv_path = base_dir + '/curated/hvfhv_data/preprocessed_hvfhv'
hvfhv_sdf = spark.read.parquet(hvfhv_path)
hvfhv_sdf.show(5)

+-----------------+--------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|   request_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|           HV0003|

In [24]:
num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 100725779
Number of columns: 22


In [25]:
hvfhv_sdf.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: double (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_flag: string (nullable = true)
 |-- wav_request_flag: string (nullable = true)
 |-- wav_match_flag: s

# Drop Unrelated Columns:

In [26]:
hvfhv_sdf = hvfhv_sdf.drop("dispatching_base_num", "access_a_ride_flag")

num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

hvfhv_sdf.show(5)

Number of rows: 100725779
Number of columns: 20
+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+
|hvfhs_license_num|   request_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|wav_request_flag|wav_match_flag|
+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+
|           HV0003|2023-07-01 00:04:21|2023-07-01 00:08:30|2023-07-01 00:33:33|          72

# Data Encoding:

We do this because the correlation heatmap only accepts numerical data.

In [27]:
num_license_num = hvfhv_sdf.select('hvfhs_license_num').distinct().count()
print(num_license_num)

2


Apply labeling encoding on `hvfhs_license_num`:

In [28]:
unique_license_num = hvfhv_sdf.select('hvfhs_license_num').distinct()
unique_license_num.show(truncate=False)

+-----------------+
|hvfhs_license_num|
+-----------------+
|HV0005           |
|HV0003           |
+-----------------+



In [29]:
# Replace 'HV0003' with 0 and 'HV0005' with 1
hvfhv_sdf = hvfhv_sdf.withColumn('hvfhs_license_num', 
                                 when(col('hvfhs_license_num') == 'HV0005', 1).otherwise(0))

In [30]:
hvfhv_sdf.show(5)

+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+
|hvfhs_license_num|   request_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|wav_request_flag|wav_match_flag|
+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+
|                0|2023-07-01 00:04:21|2023-07-01 00:08:30|2023-07-01 00:33:33|          72|          26|      4.79|    25.05|             

Apply labeling encoding on `shared_request_flag`, `shared_match_flag`, `wav_request_flag`, and `wav_match_flag`:

In [31]:
# Replace 'N' with 0 and 'Y' with 1 in the following columns
columns_to_replace = ['shared_request_flag', 'shared_match_flag', 'wav_request_flag', 'wav_match_flag']

for column in columns_to_replace:
    hvfhv_sdf = hvfhv_sdf.withColumn(column, when(col(column) == 'Y', 1).otherwise(0))

In [32]:
hvfhv_sdf.show(5)

+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+
|hvfhs_license_num|   request_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|wav_request_flag|wav_match_flag|
+-----------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+
|                0|2023-07-01 00:04:21|2023-07-01 00:08:30|2023-07-01 00:33:33|          72|          26|      4.79|    25.05|             

Confirm all columns are numeric except datetime columns:

In [33]:
hvfhv_sdf.printSchema()

root
 |-- hvfhs_license_num: integer (nullable = false)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: double (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: integer (nullable = false)
 |-- shared_match_flag: integer (nullable = false)
 |-- wav_request_flag: integer (nullable = false)
 |-- wav_match_flag: integer (nullable = false)



# Save the Preprocessed HVFHV Dataset:

In [35]:
hvfhv_dir = base_dir + '/curated/hvfhv_data'
file_name = 'preprocessed_hvfhv_2'
hvfhv_path = os.path.join(hvfhv_dir, file_name)
hvfhv_sdf.write.mode('overwrite').parquet(hvfhv_path)